# Coherencia y diversidad

Calcula la coherencia (C_npmi) y la diversidad (TD) de cada técnica y los contrastes entre técnicas.

## Rutas

Las rutas son relativas a la carpeta `notebooks/`. Los datos no forman parte del repositorio (ver README).

In [ ]:
# Comprobación de rutas
import os
PROY = os.path.abspath('..')
print("carpeta del notebook :", os.getcwd())
print("raiz del proyecto    :", PROY)
for _sub in ("Topic/Dataframes", "Topic/Resultados", "csv_parques"):
    _p = os.path.join(PROY, _sub)
    print(f"  {'OK ' if os.path.isdir(_p) else 'FALTA'}  {_sub}")
print("\nSi algo dice FALTA, el kernel no arranco en la carpeta del notebook,\n"
      "o los datos no estan donde se los espera (ver README).")

### Corrida de referencia

In [ ]:
import pickle, numpy as np
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

TOP_N = 10
BASE = "../Topic/Dataframes"
resultados = {}

def metricas(ck, label):
    d = pickle.load(open(f"{BASE}/{ck}", "rb"))
    sw = set(d['final_stopwords'])
    topic_words = [[w for w in tw if w not in sw][:TOP_N] for tw in d['topic_words_raw']]
    ref = d['ref_texts']
    dicc = Dictionary(ref)
    per = CoherenceModel(topics=topic_words, texts=ref, dictionary=dicc,
                         coherence='c_npmi').get_coherence_per_topic()
    cnpmi = float(np.nanmean(per)); n_nan = int(np.sum(np.isnan(per)))
    allw = [w for t in topic_words for w in t]
    TD = len(set(allw)) / len(allw)
    resultados[label] = (cnpmi, TD)
    print(f"[{label}] C_npmi={cnpmi:.4f} | TD={TD:.4f} | nan={n_nan} | n/tóp={[len(t) for t in topic_words]}")

for ck, l in [("ckpt_lda.pkl","LDA"), ("ckpt_tfidf.pkl","TF-IDF"),
              ("ckpt_w2v.pkl","W2V"), ("ckpt_bertopic.pkl","BERTopic")]:
    metricas(ck, l)

### Fila LLM de la Tabla 4.1 (diversidad)

La coherencia del LLM que se reporta se calcula en `09_analisis_derivados`, bloque M6, sin las entradas del diccionario en el corpus de referencia.

In [ ]:
import unicodedata, pandas as pd, spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from nltk.corpus import stopwords as nltk_stop

BASE = ".."
results = pd.read_csv(f"{BASE}/Topic/Resultados/Topico/LLM_results.csv")
df_base = pd.read_pickle(f"{BASE}/df_generic_cleaned.pkl")

def remove_accents(t):
    return ''.join(c for c in unicodedata.normalize('NFKD', str(t)) if unicodedata.category(c) != 'Mn')

CONTEXTO = {'lugar','sitio','espacio','zona','parte','parque','montevideo','uruguay','visita','visitar','paseo','pasear','ir','hacer','dia','noche','fin','semana','año','vez','momento','cosa','caso','tema','tipo','aunque','sino','pues','tal','tan','asi','ahi','alla','aca','pais','ciudad','capital','gente','persona','publico','mate','pasar','quedar','estar','ser','haber','tener','poder','deber','dar','ver','saber','querer','llegar','decir','salir','venir','encontrar','dejar','poner','parecer','llevar','seguir','etc','siempre','alguno','solo','medio','ahora','mismo'}
SENTIMIENTO = {'lindo','hermoso','bonito','bello','precioso','feo','horrible','espantoso','terrible','bueno','malo','buen','excelente','espectacular','genial','fantastico','increible','maravilloso','divino','impecable','impresionante','pesimo','lamentable','desastre','patetico','mejor','peor','mayor','menor','gran','grande','bien','mal','super','muy','bastante','poco','mucho','agradable','desagradable','recomendable','recomendado','tranquilo','paz','relax','gustar','encantar','disfrutar','recomendar','valer','ideal','perfecto','destacar','pena','disfrutable'}
ERRORES = {'lindar','exelente','exelentes','escelente','imprecionante','ermoso','ermosa','recomeir','recomiendable','recomendacion','mas'}
CORTESIA = {'gracias'}; LUGARES = {'prado'}; PROTEGER = set()

nombres_parque = set()
for nombre in df_base['nombre_parque'].str.lower().unique():
    for token in nombre.split():
        nombres_parque.add(remove_accents(token))
nombres_parque -= PROTEGER

final_stopwords = {remove_accents(w) for w in nltk_stop.words('spanish')}
for grupo in (CONTEXTO, SENTIMIENTO, ERRORES, CORTESIA, LUGARES):
    final_stopwords |= {remove_accents(w) for w in grupo}
final_stopwords |= nombres_parque
final_stopwords = sorted(final_stopwords)

nlp = spacy.load('es_core_news_sm', disable=['parser','ner'])
results['texto_lematizado'] = [' '.join(t.lemma_ for t in doc if t.is_alpha)
    for doc in nlp.pipe(results['text'].astype(str).tolist(), batch_size=200)]

TOP_N = 10
topic_words = []
for tid in sorted(results['topic_id'].unique()):
    if tid == -1: continue
    textos = results[results['topic_id'] == tid]['texto_lematizado'].tolist()
    if len([t for t in textos if t.strip()]) < 3: continue
    tfidf = TfidfVectorizer(max_features=200, min_df=2, stop_words=final_stopwords, strip_accents='unicode')
    M = tfidf.fit_transform(textos).toarray()
    nombres = tfidf.get_feature_names_out()
    idx = M.mean(axis=0).argsort()[-TOP_N:][::-1]
    topic_words.append([nombres[i] for i in idx])

textos_tokenizados = [remove_accents(t).split() for t in results['texto_lematizado']]
diccionario = Dictionary(textos_tokenizados)
cnpmi = CoherenceModel(topics=topic_words, texts=textos_tokenizados,
                       dictionary=diccionario, coherence='c_npmi').get_coherence()
allw = [w for t in topic_words for w in t]
TD = len(set(allw)) / len(allw)
print(f"[LLM]  C_npmi={cnpmi:.4f} | TD={TD:.4f}")
try: resultados['LLM'] = (cnpmi, TD)
except NameError: pass

### Tamaño del corpus de referencia (Tabla 3.1)

In [ ]:
import pickle
BASE = "../Topic/Dataframes"
for ck in ["ckpt_lda.pkl","ckpt_tfidf.pkl","ckpt_w2v.pkl","ckpt_bertopic.pkl"]:
    d = pickle.load(open(f"{BASE}/{ck}","rb"))
    tw = d['topic_words_raw']
    print(f"{ck:20} tópicos={len(tw):4} | palabras/tóp min-max={min(len(t) for t in tw)}-{max(len(t) for t in tw)} | docs_ref={len(d['ref_texts'])}")

### Bigramas y términos fuera del corpus en las palabras representativas

In [ ]:
import pickle, unicodedata
BASE = "../Topic/Dataframes"
def _qt(t): return ''.join(c for c in unicodedata.normalize('NFKD', str(t)) if unicodedata.category(c) != 'Mn')
for ck in ["ckpt_lda.pkl","ckpt_tfidf.pkl","ckpt_w2v.pkl","ckpt_bertopic.pkl"]:
    d = pickle.load(open(f"{BASE}/{ck}","rb")); sw = set(d['final_stopwords'])
    vocab = set(w for doc in d['ref_texts'] for w in doc)
    print(f"\n=== {ck} ===")
    for i, tw in enumerate(d['topic_words_raw']):
        f = [w for w in tw if w not in sw][:10]
        bg = [w for w in f if ' ' in w]
        oov = [w for w in f if _qt(w).lower() not in vocab]
        print(f"  tóp{i}: n_top10={len(f)} | bigramas={len(bg)} | fuera_del_corpus={oov}")

### Coherencia y diversidad sobre diez semillas (Tabla 4.1)

In [ ]:
# ============================================================
# MULTI-SEMILLA — TABLA DE CALIDAD  C_npmi + TD (media +/- desvio, ddof=1)
# Lee los 4 ckpt_*_multiseed.pkl. TOP_N=10. ref_texts invariante a la semilla.
# LLM = punto unico (sin semilla, clasificacion zero-shot determinista): se usa el valor
# de la celda del LLM ya existente; aca sin barra de error.
# ============================================================
import pickle, numpy as np
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

TOP_N = 10
DFDIR = "../Topic/Dataframes"

def tabla(ck, label):
    d = pickle.load(open(f"{DFDIR}/{ck}", "rb"))
    sw = set(d['final_stopwords']); ref = d['ref_texts']; dic = Dictionary(ref)
    cn, td = [], []
    for s, tw_raw in d['topic_words_raw_by_seed'].items():
        tw = [[w for w in row if w not in sw][:TOP_N] for row in tw_raw]
        per = CoherenceModel(topics=tw, texts=ref, dictionary=dic, coherence='c_npmi').get_coherence_per_topic()
        cn.append(float(np.nanmean(per)))
        allw = [w for t in tw for w in t]; td.append(len(set(allw)) / len(allw))
    n = len(cn)
    print(f"[{label:8}] C_npmi={np.mean(cn):+.4f} +/- {np.std(cn, ddof=1):.4f} | "
          f"TD={np.mean(td):.4f} +/- {np.std(td, ddof=1):.4f} | n={n}")
    if 'n_topics_by_seed' in d:
        v = list(d['n_topics_by_seed'].values())
        print(f"           topicos naturales por semilla: min={min(v)} max={max(v)}")
    return {'cnpmi': cn, 'td': td}

resultados_ms = {}
for ck, l in [("ckpt_lda_multiseed.pkl","LDA"), ("ckpt_tfidf_multiseed.pkl","TF-IDF"),
              ("ckpt_w2v_multiseed.pkl","W2V"), ("ckpt_bertopic_multiseed.pkl","BERTopic")]:
    resultados_ms[l] = tabla(ck, l)
print("\nLLM -> punto unico (sin semilla). Usar el valor de la celda del LLM. Sin barra de error.")

### Número de tópicos preferido por semilla (Tabla 3.2)

In [ ]:
# ============================================================
# MULTI-SEMILLA — ROBUSTEZ DE LA SELECCION DE K (barrido sobre 10 semillas)
# Lee barrido_{lda,tfidf,w2v}_multiseed.pkl. Reporta, por K, la media +/- desvio del
# criterio nativo, y en cuantas semillas el optimo cae en el PISO del rango (K=2).
# Defiende el hallazgo "sin optimo interior" como robusto a la semilla (Agrawal 2018).
# ============================================================
import pickle, numpy as np
DFDIR = "../Topic/Dataframes"

def resumen(ck, label, criterio, mejor):
    # criterio: 'cv' (max), 'ch' (max), 'db' (min).  mejor = 'max' o 'min'
    d = pickle.load(open(f"{DFDIR}/{ck}", "rb")); B = d['barrido']; ks = d['k_range']
    def val(cell):
        if isinstance(cell, tuple):   # (CH, DB)
            return cell[0] if criterio == 'ch' else cell[1]
        return cell                   # C_v escalar
    piso = ks[0]; en_piso = 0; args = []
    print(f"\n[{label}] criterio={label and criterio.upper()} (mejor={mejor})")
    for k in ks:
        vals = [val(B[s][k]) for s in B]
        print(f"  K={k}: {np.mean(vals):+.4f} +/- {np.std(vals, ddof=1):.4f}")
    for s in B:
        serie = {k: val(B[s][k]) for k in ks}
        best = (max if mejor == 'max' else min)(serie, key=serie.get)
        args.append(best); en_piso += (best == piso)
    print(f"  optimo por semilla: {args}")
    print(f"  -> {en_piso}/{len(B)} semillas ubican el optimo en el PISO (K={piso})")

resumen("barrido_lda_multiseed.pkl", "LDA", "cv", "max")
resumen("barrido_tfidf_multiseed.pkl", "TF-IDF", "ch", "max")
resumen("barrido_w2v_multiseed.pkl", "W2V", "ch", "max")
print("\nNota: BERTopic no tiene barrido (K emerge de HDBSCAN); ver n_topics_by_seed en la tabla de calidad.")

### Contraste pareado (Tabla 4.2)

In [ ]:
# ============================================================
# TEST PAREADO — Wilcoxon signed-rank + Holm (sobre las 10 semillas)
# Formaliza el "empate/separacion" que se ve en la tabla de calidad. Compara los 4 modelos
# multi-semilla emparejando por semilla. El LLM NO entra (no tiene semillas).
# Requiere haber corrido antes la celda "TABLA DE CALIDAD" (define resultados_ms con las
# listas de 10 valores por modelo).  Reporta diferencia mediana + p + p_Holm.
# ============================================================
import numpy as np
from scipy.stats import wilcoxon
from itertools import combinations

assert 'resultados_ms' in dir(), "Corre primero la celda TABLA DE CALIDAD (define resultados_ms)."
modelos = list(resultados_ms.keys())   # LDA, TF-IDF, W2V, BERTopic

def holm(items):
    # items: lista de (etiqueta, p) -> dict etiqueta -> p ajustada (Holm)
    m = len(items)
    order = sorted(range(m), key=lambda i: items[i][1])
    adj = [0.0]*m; prev = 0.0
    for rank, i in enumerate(order):
        v = min(1.0, max((m - rank) * items[i][1], prev)); prev = v
        adj[i] = v
    return {items[i][0]: adj[i] for i in range(m)}

for metric, nombre in [('cnpmi', 'C_npmi'), ('td', 'TD')]:
    print(f"\n===== {nombre} — Wilcoxon pareado (n=10 semillas), Holm =====")
    pares = []
    for a, b in combinations(modelos, 2):
        xa = np.asarray(resultados_ms[a][metric], float)
        xb = np.asarray(resultados_ms[b][metric], float)
        med = float(np.median(xa - xb))
        try:
            _, p = wilcoxon(xa, xb)
        except ValueError:      # todas las diferencias iguales
            p = 1.0
        pares.append((f"{a} vs {b}", p, med))
    hp = holm([(l, p) for l, p, _ in pares])
    for l, p, med in sorted(pares, key=lambda t: t[1]):
        veredicto = 'DISTINGUIBLES' if hp[l] < 0.05 else 'no distinguibles'
        print(f"  {l:20} | dif.mediana={med:+.4f} | p={p:.4f} | p_Holm={hp[l]:.4f} | {veredicto}")
print("\nLectura: 'no distinguibles' = las bandas se pisan, no podes afirmar que uno gane.")

### Semilla 42 frente a la media (Sección 4.2.1)

In [ ]:
for m in ['TF-IDF', 'LDA', 'BERTopic', 'W2V']:
    v = resultados_ms[m]['cnpmi']
    print(f"{m:9} seed42={v[0]:+.4f} | media={sum(v)/len(v):+.4f} | delta={v[0]-sum(v)/len(v):+.4f}")

### Unigramas disponibles por tópico

In [ ]:
import pickle, numpy as np
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

DFDIR = "../Topic/Dataframes"
CKPTS = [("ckpt_lda_multiseed.pkl","LDA"), ("ckpt_tfidf_multiseed.pkl","TF-IDF"),
         ("ckpt_w2v_multiseed.pkl","W2V"), ("ckpt_bertopic_multiseed.pkl","BERTopic")]

minimo = 99
for ck, label in CKPTS:
    d = pickle.load(open(f"{DFDIR}/{ck}", "rb")); sw = set(d['final_stopwords'])
    for s, tw_raw in d['topic_words_raw_by_seed'].items():
        filt = [[w for w in row if w not in sw] for row in tw_raw]
        uni  = [len([w for w in row if ' ' not in w]) for row in filt]
        minimo = min(minimo, min(uni))
        if s == 42:
            bg = [sum(1 for w in row[:10] if ' ' in w) for row in filt]
            print(f"\n[{label} seed42] bigramas en el top-10 por tópico={bg} | unigramas disponibles={uni}")
            for i, row in enumerate(filt):
                print(f"   tóp{i}: {row[:10]}")
print(f"\n==> N común máximo de unigramas: {minimo}")

### Coherencia restringida a unigramas (Sección 4.2.1)

In [ ]:
def tabla_unigramas(ck, label, N):
    d = pickle.load(open(f"{DFDIR}/{ck}", "rb"))
    sw = set(d['final_stopwords']); ref = d['ref_texts']; dic = Dictionary(ref)
    cn, td = [], []
    for s, tw_raw in d['topic_words_raw_by_seed'].items():
        tw = [[w for w in row if w not in sw and ' ' not in w][:N] for row in tw_raw]
        per = CoherenceModel(topics=tw, texts=ref, dictionary=dic,
                             coherence='c_npmi').get_coherence_per_topic()
        cn.append(float(np.nanmean(per)))
        allw = [w for t in tw for w in t]; td.append(len(set(allw))/len(allw))
    print(f"[{label:8}] N={N} | C_npmi={np.mean(cn):+.4f} +/- {np.std(cn, ddof=1):.4f} | "
          f"TD={np.mean(td):.4f} +/- {np.std(td, ddof=1):.4f}")

N = minimo
for ck, l in CKPTS:
    tabla_unigramas(ck, l, N)